# Analyse du churn — abonnements de services numériques

**Auteure :** Danielle OUATTARA · **Données :** [Telco Customer Churn (Kaggle)](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)

**Question :** quels clients risquent de résilier, pourquoi, et combien de revenu mensuel est en jeu ?

| Étape | Contenu | Alternative sans Python |
|---|---|---|
| 1 | Import | Power Query |
| 2 | Nettoyage | Power Query |
| 3 | Analyse exploratoire | `sql/churn_queries.sql` |
| 4 | Visualisation | Dashboard Power BI |
| 5 | Conclusion | — |

> Mode d'emploi : exécuter les cellules une par une avec **Maj + Entrée**, de haut en bas.

## 1. Import des bibliothèques et des données

In [ ]:
import pandas as pd                      # pandas : la bibliothèque qui manipule des tableaux (comme Excel)
import matplotlib.pyplot as plt          # matplotlib : la bibliothèque qui dessine les graphiques

pd.set_option("display.max_columns", None)   # afficher toutes les colonnes au lieu d'en masquer une partie

df = pd.read_csv("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")  # lit le CSV et le range dans un tableau nommé df

In [ ]:
print(df.shape)                          # affiche (nombre de lignes, nombre de colonnes) : on attend (7043, 21)
df.head()                                # affiche les 5 premières lignes pour voir à quoi ressemblent les données

In [ ]:
df.info()                                # liste chaque colonne avec son type : on repère que TotalCharges est du texte (object)

## 2. Nettoyage
`TotalCharges` est stockée en texte car 11 clients (ancienneté = 0 mois) ont une valeur vide.

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")  # convertit en nombre ; les vides deviennent NaN (valeur manquante)
print(df["TotalCharges"].isna().sum())                                    # compte les valeurs manquantes : on attend 11
df["TotalCharges"] = df["TotalCharges"].fillna(0)                         # remplace ces manquants par 0 (clients pas encore facturés)

In [ ]:
df["churn_flag"] = (df["Churn"] == "Yes").astype(int)   # crée une colonne 1 si le client est parti, 0 sinon

In [ ]:
bornes = [-1, 6, 12, 24, 48, 100]                        # limites des tranches d'ancienneté, en mois
etiquettes = ["0-6 mois", "7-12 mois", "13-24 mois", "25-48 mois", "49 mois et +"]  # nom de chaque tranche
df["tranche_anciennete"] = pd.cut(df["tenure"], bins=bornes, labels=etiquettes)    # range chaque client dans sa tranche

In [ ]:
options = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport"]  # les 4 options de sécurité/support
df["nb_options_securite"] = (df[options] == "Yes").sum(axis=1)                  # compte, ligne par ligne, les « Yes » parmi ces 4 colonnes

In [ ]:
print(df.duplicated(subset="customerID").sum())   # vérifie qu'aucun client n'apparaît deux fois : on attend 0

## 3. Analyse exploratoire (EDA)
### 3.1 KPI globaux

In [ ]:
taux_churn = df["churn_flag"].mean()                                  # la moyenne d'une colonne 0/1 = le taux de churn
mrr_total = df["MonthlyCharges"].sum()                                # revenu mensuel total de la base
mrr_perdu = df.loc[df["churn_flag"] == 1, "MonthlyCharges"].sum()     # revenu mensuel des seuls clients partis

print(f"Taux de churn : {taux_churn:.1%}")                            # :.1% = format pourcentage, 1 décimale
print(f"MRR total     : {mrr_total:,.0f} $")                          # :,.0f = séparateur de milliers, 0 décimale
print(f"MRR perdu     : {mrr_perdu:,.0f} $ ({mrr_perdu / mrr_total:.1%} du MRR)")

### 3.2 Churn par dimension

In [ ]:
def taux_par(colonne):                                   # définit une petite fonction réutilisable
    return (df.groupby(colonne, observed=True)["churn_flag"]   # regroupe les clients par valeur de la colonne
              .agg(nb_clients="count", taux_churn="mean")      # compte les clients et calcule le taux de churn par groupe
              .sort_values("taux_churn", ascending=False))     # trie du plus risqué au moins risqué

In [ ]:
taux_par("Contract")                     # churn par type de contrat

In [ ]:
taux_par("tranche_anciennete")           # churn par tranche d'ancienneté

In [ ]:
taux_par("PaymentMethod")                # churn par mode de paiement

In [ ]:
taux_par("nb_options_securite")          # churn selon le nombre d'options de sécurité souscrites

### 3.3 Matrice de priorisation : contrat × ancienneté

In [ ]:
matrice = pd.pivot_table(df,                         # tableau croisé dynamique, comme dans Excel
                         index="Contract",           # en lignes : le type de contrat
                         columns="tranche_anciennete",  # en colonnes : la tranche d'ancienneté
                         values="churn_flag",        # la valeur analysée
                         aggfunc="mean",             # la moyenne = le taux de churn
                         observed=True)              # n'affiche que les combinaisons existantes
matrice.style.format("{:.0%}").background_gradient(cmap="Reds")  # affiche en % avec un dégradé de rouge

## 4. Visualisation

In [ ]:
churn_contrat = taux_par("Contract")["taux_churn"]       # récupère uniquement la colonne des taux

fig, ax = plt.subplots(figsize=(7, 4))                   # crée une zone de dessin de 7 × 4 pouces
churn_contrat.plot(kind="bar", ax=ax, color="#C0392B")   # dessine un histogramme rouge
ax.set_title("Taux de churn par type de contrat")        # titre du graphique
ax.set_ylabel("Taux de churn")                           # libellé de l'axe vertical
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))  # axe vertical en %
plt.xticks(rotation=0)                                   # libellés de l'axe horizontal à l'horizontale
plt.tight_layout()                                       # ajuste les marges
plt.savefig("../visuals/churn_par_contrat.png", dpi=150) # enregistre l'image dans le dossier visuals
plt.show()                                               # affiche le graphique

In [ ]:
churn_anc = taux_par("tranche_anciennete")["taux_churn"].sort_index()   # sort_index remet les tranches dans l'ordre

fig, ax = plt.subplots(figsize=(7, 4))
churn_anc.plot(kind="line", marker="o", ax=ax, color="#2C3E50")         # courbe avec un point par tranche
ax.set_title("Le risque de churn baisse avec l'ancienneté")
ax.set_ylabel("Taux de churn")
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
plt.tight_layout()
plt.savefig("../visuals/churn_par_anciennete.png", dpi=150)
plt.show()

## 5. Conclusion
> *À rédiger après exécution — 3 constats chiffrés + 3 recommandations.*

1. **Constat 1 :** les contrats mensuels churnent à `[XX] %`, contre `[X] %` pour les contrats 2 ans → …
2. **Constat 2 :** `[XX] %` des départs ont lieu dans les 6 premiers mois → …
3. **Constat 3 :** les clients sans option de sécurité churnent `[X]` fois plus → …

**Recommandations :** voir le README, section 7.